# Phenotypic activity with CoPairs (mAP)

This notebook computes **phenotypic activity** using the **CoPairs** framework: we measure how well replicate profiles of the same perturbation retrieve each other (vs. other perturbations) using **average precision (AP)**, and then summarize per-perturbation using **mean average precision (mAP)**.

**Interpretation (high level)**  
- Higher **mAP** → stronger and more consistent phenotypic signature.  
- We estimate a **p-value** by comparing each perturbation to a null distribution derived from the negative control.

> Note: since we test only a few compounds, a cutoff like **p < 0.05** is usually fine.  
> If you later scale up to many perturbations, consider reporting **FDR** as well.


## 0) Setup

Edit the parameters below:
- `PROFILE_PATH`: path to your *per-well* profile table (one row per well).
- `NEGCON_TREATMENT`: how to define the negative control condition.
- `ALPHA`: significance threshold for calling "active".


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from copairs import map
from copairs.matching import assign_reference_index
from copairs.map.average_precision import p_values

# Optional (recommended) for nicer label placement
# If you don't have it installed: pip install adjustText
from adjustText import adjust_text


In [ ]:
# -----------------------------
# Parameters (edit as needed)
# -----------------------------
PROFILE_PATH = "/Users/marcelobispojesus/Documents/dev/keila/lcp-inhibitors-huh7/workspace/profiles/per_well_features_selected_merged.parquet"

# Negative control definition (adapt if needed)
NEGCON_CONTROL_TYPE = "negcon"
NEGCON_TREATMENT = "Non-treated"

# Stats / runtime
ALPHA = 0.05
NULL_SIZE = 200_000     # increase (e.g., 1_000_000) for more stable p-values, slower
SEED = 0

# If you want to exclude a specific condition from reporting (e.g., the negcon itself)
EXCLUDE_SAMPLE_IDS = {f"{NEGCON_TREATMENT}__0.0"}  # adjust dose string if your negcon dose differs


## 1) Load profiles

We expect a **per-well** table with:
- `Metadata_*` columns (plate, well, treatment, concentration, control_type, etc.)
- numeric feature columns (CellProfiler / profiling features)


In [ ]:
df = pd.read_parquet(PROFILE_PATH, engine="pyarrow")
df.shape


## 2) Sanity checks

We confirm this is *per-well* (one row per Plate+Well).  
If your table was originally site-level, you would aggregate before running CoPairs — but here we expect aggregation is already done.


In [ ]:
# Per-well check: max must be 1
well_counts = df.groupby(["Metadata_Plate", "Metadata_Well"]).size()
well_counts.describe()


## 3) Define a perturbation label (treatment + concentration)

CoPairs needs a label that uniquely identifies a condition.  
We create `Metadata_sample_id = Treatment__Concentration` so different doses are treated as distinct perturbations.


In [ ]:
df = df.copy()

# This column is often unnecessary at the per-well level; keep or drop as you prefer
df = df.drop(columns=["Metadata_Site"], errors="ignore")

df["Metadata_sample_id"] = (
    df["Metadata_Treatment"].astype(str)
    + "__"
    + df["Metadata_Concentration"].astype(str)
)

# Metadata columns (for pairing logic)
metadata_cols = [c for c in df.columns if c.startswith("Metadata_")]
meta = df.loc[:, metadata_cols]

# Ensure there are no duplicated metadata column names
assert not meta.columns.duplicated().any(), "Duplicated metadata columns found (check your selections)."

meta.groupby("Metadata_sample_id").size().sort_values()


## 4) Define the negative control and reference index

We use a **negative control query** to define the null baseline.

`assign_reference_index` assigns a `Metadata_reference_index` so that comparisons are constrained within a relevant context (often **plate/batch**).


In [ ]:
negcon_query = (
    f"(Metadata_Control_Type == '{NEGCON_CONTROL_TYPE}') & (Metadata_Treatment == '{NEGCON_TREATMENT}')"
)

reference_col = "Metadata_reference_index"
df_activity = assign_reference_index(
    df,
    negcon_query,
    reference_col=reference_col,
    default_value=-1,
)

# Inspect whether reference indices were assigned as expected
df_activity[reference_col].value_counts().head(20), df_activity.query(negcon_query)[reference_col].value_counts().head(20)


## 5) Build the profile matrix (features only)

We extract numeric features, drop columns with NaN/Inf, and remove constant features (no variance).


In [ ]:
metadata = df_activity.filter(regex="^Metadata").copy()

feature_df = df_activity.filter(regex="^(?!Metadata)").select_dtypes(include="number")
feature_df = feature_df.replace([np.inf, -np.inf], np.nan).dropna(axis=1)
feature_df = feature_df.loc[:, feature_df.nunique() > 1]

profiles = feature_df.to_numpy(dtype=np.float32)

metadata.shape, profiles.shape


## 6) Compute AP, p-values, and mAP

**Pair definitions**
- Positive pairs: same `Metadata_sample_id` within the same `Metadata_reference_index`.
- Negative pairs: different `Metadata_sample_id` within the same `Metadata_reference_index`.

Then we:
1) compute **AP** for each well,
2) compute a **p-value** against the negative-control null,
3) summarize per perturbation into **mAP**.


In [ ]:
pos_sameby = ["Metadata_sample_id", reference_col]
pos_diffby = []

neg_sameby = [reference_col]
neg_diffby = ["Metadata_sample_id"]

activity_ap = map.average_precision(
    metadata, profiles,
    pos_sameby, pos_diffby,
    neg_sameby, neg_diffby,
)

# Drop explicit negcon condition from reporting (optional)
if EXCLUDE_SAMPLE_IDS:
    activity_ap = activity_ap[~activity_ap["Metadata_sample_id"].isin(EXCLUDE_SAMPLE_IDS)].copy()

activity_ap["p_value"] = p_values(activity_ap, null_size=NULL_SIZE, seed=SEED)

activity_map = map.mean_average_precision(
    activity_ap,
    pos_sameby,
    null_size=NULL_SIZE,
    threshold=ALPHA,
    seed=SEED,
)

activity_map.sort_values("mean_average_precision", ascending=False)


## 7) Results table (significant hits)

A "hit" here means **p < ALPHA**.


In [ ]:
df_res = activity_map.copy()
df_res = df_res.sort_values(["p_value", "mean_average_precision"], ascending=[True, False])

hits = df_res[df_res["p_value"] < ALPHA].copy()
hits


## 8) Phenotypic activity plot (mAP vs -log10(p))

- x-axis: **mAP**  
- y-axis: **-log10(p-value)**  
- horizontal line: **p = ALPHA** cutoff  
- labels are automatically adjusted using **adjustText** to avoid overlaps.


In [ ]:
dfp = df_res.copy()
dfp["neglog10_p"] = -np.log10(dfp["p_value"].clip(lower=1e-300))
dfp["is_hit"] = dfp["p_value"] < ALPHA

label_col = "Metadata_sample_id"

plt.figure(figsize=(10, 6))
plt.scatter(dfp["mean_average_precision"], dfp["neglog10_p"], s=35, alpha=0.85)
plt.axhline(-np.log10(ALPHA), linewidth=1)

texts = []
for _, r in dfp[dfp["is_hit"]].iterrows():
    texts.append(
        plt.text(
            r["mean_average_precision"],
            r["neglog10_p"],
            str(r[label_col]),
            fontsize=8
        )
    )

adjust_text(texts, arrowprops=dict(arrowstyle="-", lw=0.5))
plt.xlabel("mAP (mean average precision)")
plt.ylabel("-log10(p-value)")
plt.title(f"Phenotypic activity (cutoff p<{ALPHA})")
plt.tight_layout()
plt.show()
